# Thumbs_Robot: Unitree G1 Three-Digit Hand-Gesture Control via PPO
## Notebook Handbook & Grading Guide

This notebook serves as the guidebook and hands-on manual for the CSCN8020 Reinforcement Learning final project. You can run the cells step-by-step to complete the entire pipeline: environment setup, model auditing, environment validation, model training, and results visualization.

### Project Overview & Team
- **Project Goal**: Control the Unitree G1 robot's 3-digit hand morphology (comprising two main fingers and one thumb) using a continuous-action Actor-Critic PPO controller. The controller must achieve stable, smooth, and coordinated movements for three target gestures in the MuJoCo simulation environment: Thumbs Up, Open/Stop, and Thumbs Down.
- **Academic Highlights**: Implementation of the **"Math MDP $\rightarrow$ Algorithm Logic $\rightarrow$ Code Variables $\rightarrow$ Real-time logs" 4-in-1 alignment mapping** as required by the grading criteria.
- **Team Members**: Emmanuel • Liggia • Cemil • Chao

### 1. Environment Setup
To ensure this project can be fully reproduced on other machines, please execute the following cells in order. This will upgrade pip, install dependency packages, clone the official third-party Unitree repository, and verify the Python environment setup.

#### Step 0: Open Project Folder with WSL and Setup Virtual Environment

1. Press Ctrl + Shift + P in your IDE (Open Command Palette).
2. Type and select: WSL: Connect to WSL
3. Click "Open Folder" and choose your project directory inside WSL (e.g. /mnt/l/Reinforcement Learning Programming/Final_Project)
4. Confirm status: Verify that the bottom left corner of the IDE shows "WSL: Ubuntu" (or green bar).
5. Before installing MuJoCo or Unitree software, install the Linux packages required for Python virtual environments, C/C++ compilation, CMake/Ninja builds, OpenGL rendering, GLFW window management, and WSLg graphics.
```bash
sudo apt update && sudo apt install -y \
    python3-venv \
    python3-dev \
    build-essential \
    git \
    cmake \
    ninja-build \
    pkg-config \
    libglfw3 \
    libglfw3-dev \
    libgl1-mesa-dev \
    libegl1-mesa-dev \
    libxinerama-dev \
    libxcursor-dev \
    libxrandr-dev
```

6. Create a project-local virtual environment so this workshop does not interfere with other Python installations.

```bash
cd /mnt/c/Final_Project

python3 -m venv .venv
source .venv/bin/activate
```

#### Step 1: Upgrade pip and Install Dependencies from requirements.txt

In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!pip install -r requirements.txt

#### Step 2: Clone Official Third-Party Unitree MuJoCo Repository (if external/unitree_mujoco does not exist)

In [ ]:
import os
external_dir = os.path.join("external", "unitree_mujoco")
if not os.path.exists(external_dir):
    print("Cloning unitree_mujoco repository...")
    os.makedirs("external", exist_ok=True)
    !git clone https://github.com/unitreerobotics/unitree_mujoco.git {external_dir}
else:
    print(f"'{external_dir}' already exists, skipping clone.")

#### Step 3: Verify Successful Installation of Key Libraries

In [ ]:
import sys
print(f"Python Version: {sys.version}")
try:
    import mujoco
    print(f"MuJoCo Version: {mujoco.__version__}")
except ImportError:
    print("Error: MuJoCo is not installed.")
try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
except ImportError:
    print("Error: PyTorch is not installed.")
try:
    import gymnasium as gym
    print(f"Gymnasium Version: {gym.__version__}")
except ImportError:
    print("Error: Gymnasium is not installed.")

### 2. Model Joint Audit and Gesture Angle Calibration [Phase 0]
Before beginning reinforcement learning, we must generate the 29-DOF fixed-base G1 model, inspect its wrist and finger actuators, and calibrate the target joint angle vectors for the three gestures.
- **Executable Files**: [create_fixed_base_g1.py](./src/create_fixed_base_g1.py) and [g1_model_audit.py](./src/g1_rl/g1_model_audit.py)
- **Tasks**: Load the G1 robot hand XML model, print all joint and actuator names along with their limits, and calibrate/save the target joint angle vectors for the three gestures:
  1. **Thumbs Up**: Thumb extended, two fingers flexed, wrist oriented upwards.
  2. **Open/Stop**: All three digits fully extended, palm facing forward.
  3. **Thumbs Down**: Thumb extended, two fingers flexed, wrist oriented downwards.

> **3-Digit Hand Rule**: Unitree G1 uses a **2 main fingers + 1 thumb** three-digit hand morphology. Do not use 5-digit human hand structures or rendering references as evidence of implementation.

In [ ]:
# Generate fixed-base G1 model and execute model joint audit
!python src/create_fixed_base_g1.py
!python src/g1_rl/g1_model_audit.py --xml-path assets/g1_fixed_base/scene_29dof_fixed_base.xml --no-viewer

### 3. Gymnasium Environment Development and Random Action Testing [Phase 1]
- **Executable File**: [g1_hand_env.py](./src/g1_rl/g1_hand_env.py)
- **任務**：
  * Define continuous state space $s_t$: $s_t = [q_t, \dot{q}_t, q_{\text{target}}(g), q_{\text{target}}(g)-q_t, \text{one\_hot}(g), a_{t-1}]$
  * Define continuous action space $a_t$: Joint position increments for wrist and fingers.
  * Implement reward function:
    $$r_t = w_p(e_{t-1} - e_t) - w_h E_{\text{hand}} - w_o E_{\text{orientation}} - w_v \|\dot{q}_t\|^2 - w_a \|a_t\|^2 - w_s \|a_t-a_{t-1}\|_2^2 + b_{\text{hold}} I_{\text{hold}} - c_{\text{time}}$$
  * Implement hold-on success conditions (Hold verification mechanism: success is declared if the pose/orientation error stays below threshold for at least 15 steps).

Please run the random action test cell below to verify that the Gymnasium wrapper in [g1_hand_env.py](./src/g1_rl/g1_hand_env.py) functions correctly:

In [ ]:
# Verify Gymnasium env initialization and execution step (random action smoke test)
import os
import sys
sys.path.append(os.path.abspath("src"))
from g1_rl.g1_hand_env import G1HandEnv

try:
    # Instantiate environment; functions as API test if model xml is being developed
    env = G1HandEnv(xml_path="assets/g1_fixed_base/scene_29dof_fixed_base.xml")
    obs, info = env.reset()
    print("SUCCESS: Environment initialized successfully!")
    print(f"Observation Space Shape: {obs.shape}")
    print(f"Action Space Shape: {env.action_space.shape}")
    print(f"Initial target gesture: {info.get('target_gesture')}")
    
    # Step simulation with a random action
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, step_info = env.step(action)
    print("SUCCESS: Environment stepped successfully!")
    print(f"Step Reward: {reward}")
    print(f"Next Obs Shape: {next_obs.shape}")
    print(f"Reward info components: {step_info.get('reward_info')}")
except Exception as e:
    print("Environment test failed. (This is normal if xml model or logic is in placeholder status)")
    print("Error details:", e)

### 4. PPO Algorithm Architecture and Network Design [Phase 2]
- **Executable Files**: [actor_critic_network.py](./src/Thumbs_Robot/actor_critic_network.py), [rollout_buffer.py](./src/Thumbs_Robot/rollout_buffer.py), [agent.py](./src/Thumbs_Robot/agent.py)

- **The table below shows the mapping between mathematical concepts, algorithm logic, code variables, and console/file logs**:
  

| 數學概念 (MDP Formula) | 演算法邏輯 (Algorithm) | 程式碼變數 (Code Variable) | Console/CSV 日誌欄位 (Log Field) |
| :--- | :--- | :--- | :--- |
| Current State $s_t$ | Observation Vector `env._get_obs()` | `obs` / `state` | `state_error_norm`, `gesture` |
| Policy Distribution $\pi_\theta(a \mid s)$ | Mean $\mu$ and Std $\sigma$ outputs | `action_dist`, `mu`, `std` | `actor_mean`, `actor_std` |
| Selected Action $a_t$ | Action sampling & boundary clipping | `raw_action`, `clipped_action` | `action_sample`, `action_clipped` |
| State Transition $p(s' \mid s, a)$ | Simulator physics steps | `next_obs`, `reward`, `terminated` | `V(s_t+1)`, `reward_total` |
| Accumulated Return $G_t$ | Compound reward computation | `reward_total`, `reward_components` | `reward_total`, `progress`, `pose`, `hold` |
| State Value $V_\phi(s)$ | Critic expected reward estimate | `value_t`, `next_value` | `V(s_t)` |
| TD Target $y_t$ | Bellman target estimation | `td_target` | `td_target` |
| Advantage Estimate $\hat{A}_t$ | GAE / TD-error computation | `advantage` | `advantage` |
| Policy Loss $L_{\text{actor}}$ | PPO clipped policy loss objective | `actor_loss` | `actor_loss` |
| Value Loss $L_{\text{critic}}$ | MSE TD-error value loss | `critic_loss` | `critic_loss` |

- **Component Responsibilities**:
  1. **Actor Network**: Takes $s_t$ as input, predicts the mean and standard deviation of the continuous action distribution, and samples actions using a Gaussian distribution.
  2. **Critic Network**: Takes $s_t$ as input, predicts the expected state value $V(s_t)$.
  3. **Rollout Buffer**: Stores on-policy trajectory transitions (States, Actions, Log_probs, Rewards, Values, Terminals) and computes GAE advantages.

### 5. PPO Components Verification [Phase 3]
- **Executable File**: [smoke_test.py](./src/Thumbs_Robot/smoke_test.py)
- **任務**：
  Verify PyTorch network forward pass (predicting Gaussian mean and std, along with state value $V$), GAE advantage estimation, and PPO clipped updates.
  Run the smoke test script below to verify neural network dimensions, buffer sampling, and optimizer updates without NaN/Inf issues.

In [ ]:
# Run smoke test to verify correctness of PPO components and network architecture
!python src/Thumbs_Robot/smoke_test.py

### 6. Math-to-Code-to-Log Mapping [Phase 4]
- **Tasks**: Implement structured CSV log output formats for transitions and updates, printing aligned formulas to the console, and verifying the "Math MDP $\rightarrow$ Algorithm $\rightarrow$ Code $\rightarrow$ Logs" 4-in-1 alignment mapping.
- **Verification Method**: Run the smoke update test, observe the update-level table printed in the console (containing algorithm metrics such as `Loss_A`, `Loss_C`, and `Entropy`, along with gesture summary), and check the generated `episode_log.csv` under `results/ppo_config_a/` (containing episode details like `Reward`, `Final Pose Error`, `Hold Duration`, and `Safety Violation`) to confirm matching variables.


In [ ]:
# Core configuration variable: modify this variable to switch experiment configs (e.g. "ppo_config_b", "ppo_config_c", etc.)
CONFIG_NAME = "ppo_config_a"

# Run update smoke test to inspect PPO console and file logs mapping output
!python src/Thumbs_Robot/train_thumbs.py --smoke-test --results-dir results/{CONFIG_NAME}


In [ ]:
# Load and display the generated episode_log.csv detailed records
import os
import pandas as pd

episode_log_path = f"results/{CONFIG_NAME}/episode_log.csv"
if os.path.exists(episode_log_path):
    df = pd.read_csv(episode_log_path)
    print(f"SUCCESS: Read {len(df)} episode details. The latest 10 rows are:")
    display(df.tail(10))
else:
    print(f"Error: Log not found at {episode_log_path}. Please execute the training cell above first.")


### 7. Target-Conditioned Headless Training [Phase 5]
- **Executable File**: [train_thumbs.py](./src/Thumbs_Robot/train_thumbs.py)
- **Tasks**: Formally launch the Unified Target-Conditioned Policy PPO training process to learn Thumbs Up, Open/Stop, and Thumbs Down gestures simultaneously.
- **Description**: Training log files and policy checkpoints will be saved periodically in `results/` and `models/` directories.

In [ ]:
# Start formal continuous-action PPO multi-gesture training (default 100,000 steps)
# Training logs will be written to the results/ directory
!python src/Thumbs_Robot/train_thumbs.py --results-dir results/{CONFIG_NAME}


In [ ]:
# Load and display the generated episode_log.csv detailed records
import os
import pandas as pd

episode_log_path = f"results/{CONFIG_NAME}/episode_log.csv"
if os.path.exists(episode_log_path):
    df = pd.read_csv(episode_log_path)
    print(f"SUCCESS: Read {len(df)} episode details. The latest 10 rows are:")
    display(df.tail(10))
else:
    print(f"Error: Log not found at {episode_log_path}. Please execute the training cell above first.")


### 8. Evaluation and 3D Visualization [Phase 6]
- **Executable Files**: [evaluate_thumbs.py](./src/Thumbs_Robot/evaluate_thumbs.py) and [render_thumbs.py](./src/Thumbs_Robot/render_thumbs.py)
- **Tasks**: Load the optimal checkpoint weights, perform deterministic greedy evaluation on benchmark tasks, and launch the interactive MuJoCo 3D visualization renderer to display dynamic gesture control.

In [ ]:
# Execute policy evaluation and success rate statistics
!python src/Thumbs_Robot/evaluate_thumbs.py --checkpoint models/{CONFIG_NAME}_best.pt --output_dir results/{CONFIG_NAME}_evaluation


### 9. Data Visualization & Delivery [Phase 7]
- **Executable File**: [plot_results.py](./src/Thumbs_Robot/plot_results.py)
- **Tasks**: Generate and plot Accumulated Return, Success Rate, Loss curves, and Policy Entropy trends.
Run the cell below to plot the training results and display them directly in the notebook.

In [ ]:
# Generate and save four independent training metric charts
!python src/Thumbs_Robot/plot_results.py --dir results/{CONFIG_NAME} --eval_csv results/{CONFIG_NAME}_evaluation/eval_results.csv

# Display the four generated metric charts directly inside the notebook
import os
from IPython.display import Image, display

img_dir = f"results/img/{CONFIG_NAME}"
plot_files = ["accumulated_returns.png", "success_rate.png", "optimization_losses.png", "policy_entropy.png"]

for plot_file in plot_files:
    plot_path = os.path.join(img_dir, plot_file)
    if os.path.exists(plot_path):
        print(f"Displaying {plot_file}:")
        display(Image(filename=plot_path))
    else:
        print(f"Warning: {plot_file} not found under {img_dir}. Check if plot_results.py ran successfully.")
